# Querying ERA5 with SQL

Edit the query, check its plan, then run it on the deployed lambdas.

In [ ]:
import os

from zarr_datafusion_lambda import sql

# Where ERA5 and the workers are. Read once a kernel, at the first query.
os.environ["AWS_REGION"] = "us-east-1"

tables = {"era5": {"url": "s3://earthmover-icechunk-era5/icechunkV2", "group": "single/spatial"}}

In [ ]:
query = """
SELECT date_trunc('day', CAST(valid_time AS TIMESTAMP))                 AS day,
     sum(t2m * cos(radians(latitude))) / sum(cos(radians(latitude))) AS mean_t2m
FROM era5
WHERE valid_time >= TIMESTAMP '2020-01-01' AND valid_time < TIMESTAMP '2020-01-08'
GROUP BY 1
ORDER BY 1
"""

In [ ]:
print(sql.explain(query, tables))

In [ ]:
result = sql.run(query, tables, progress=True)
df = result.table.to_pandas()
print(f"{result.chunks} chunks in {result.invocations} invocations")
df